# CH25H-COL5A1-TREM2 Dataset Preprocessing

Combines new 10x data with existing COL5A1 WT/CKO cells, then applies:
1. **SoupX** - ambient RNA correction (per sample, using raw + filtered H5)
2. **scDblFinder** - doublet detection (per sample)
3. **QC filtering** - min_genes, max %mito
4. Merges with existing WT/CKO from `xxo_xxt_xyt_xyo_cko_wt` files
5. Exports to internal format (mtx, gene_names, metadata)

## Sample Layout
| Dir Sample | orig.ident | Type | Experiment |
|---|---|---|---|
| S1_81_WT_1 | S1_81_WT | WT | COL5A1_KO |
| S2_83_WT_2 | S2_83_WT | WT | COL5A1_KO |
| S3_91_WT_3 | S3_91_WT | WT | COL5A1_KO |
| S4_00_CKO_1 | S4_00_CKO | COL5A1_KO | COL5A1_KO |
| S5_82_CKO_2 | S5_82_CKO | COL5A1_KO | COL5A1_KO |
| S6_96_CKO_3 | S6_96_CKO | COL5A1_KO | COL5A1_KO |
| S7_WT_1 | S7_WT | WT | GAS |
| S8_WT_2 | S8_WT | WT | GAS |
| S9_WT_3 | S9_WT | WT | GAS |
| S10_CH25H_1 | S10_CH25H | CH25H_KO | GAS |
| S11_CH25H_2 | S11_CH25H | CH25H_KO | GAS |
| S12_CH25H_3 | S12_CH25H | CH25H_KO | GAS |
| S13_Trem2_1 | S13_Trem2 | TREM2_KO | GAS |
| S14_Trem2_2 | S14_Trem2 | TREM2_KO | GAS |
| S15_Trem2_3 | S15_Trem2 | TREM2_KO | GAS |
| *(from xxo file)* | S15_82_WT | WT | COL5A1_KO |
| *(from xxo file)* | S16_84_CKO | COL5A1_KO | COL5A1_KO |

---
## 1. Setup

In [15]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.sparse import csc_matrix, csr_matrix
import anndata as ad
import scanpy as sc
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
import warnings

warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

DATA_DIR = Path('data')
CH25H_DIR = DATA_DIR / 'CH25H-COL5A1-TREM2'
print(f'Data directory: {DATA_DIR.resolve()}')
print(f'CH25H directory: {CH25H_DIR.resolve()}')

Data directory: /home/scumpia-mrl/Desktop/Sujit/Projects/mice-skin-atlas/integration_scvi/data
CH25H directory: /home/scumpia-mrl/Desktop/Sujit/Projects/mice-skin-atlas/integration_scvi/data/CH25H-COL5A1-TREM2


In [16]:
# Load R packages once at startup
ro.r("""
suppressPackageStartupMessages({
    library(SoupX)
    library(Seurat)
    library(scDblFinder)
    library(SingleCellExperiment)
    library(Matrix)
    library(BiocParallel)
})
register(SerialParam())  # deterministic single-threaded execution
""")
print('R packages loaded.')

R packages loaded.


---
## 2. Sample Configuration

In [17]:
# Map directory sample name -> metadata columns
SAMPLE_META = {
    # COL5A1 KO experiment
    'S1_81_WT_1':  {'orig.ident': 'S1_81_WT',  'Sample': 'S1',  'Type': 'WT',        'Mice': '81', 'Experiment': 'COL5A1_KO'},
    'S2_83_WT_2':  {'orig.ident': 'S2_83_WT',  'Sample': 'S2',  'Type': 'WT',        'Mice': '83', 'Experiment': 'COL5A1_KO'},
    'S3_91_WT_3':  {'orig.ident': 'S3_91_WT',  'Sample': 'S3',  'Type': 'WT',        'Mice': '91', 'Experiment': 'COL5A1_KO'},
    'S4_00_CKO_1': {'orig.ident': 'S4_00_CKO', 'Sample': 'S4',  'Type': 'COL5A1_KO', 'Mice': '00', 'Experiment': 'COL5A1_KO'},
    'S5_82_CKO_2': {'orig.ident': 'S5_82_CKO', 'Sample': 'S5',  'Type': 'COL5A1_KO', 'Mice': '82', 'Experiment': 'COL5A1_KO'},
    'S6_96_CKO_3': {'orig.ident': 'S6_96_CKO', 'Sample': 'S6',  'Type': 'COL5A1_KO', 'Mice': '96', 'Experiment': 'COL5A1_KO'},
    # GAS experiment
    'S7_WT_1':     {'orig.ident': 'S7_WT',     'Sample': 'S7',  'Type': 'WT',        'Mice': '1',  'Experiment': 'GAS'},
    'S8_WT_2':     {'orig.ident': 'S8_WT',     'Sample': 'S8',  'Type': 'WT',        'Mice': '2',  'Experiment': 'GAS'},
    'S9_WT_3':     {'orig.ident': 'S9_WT',     'Sample': 'S9',  'Type': 'WT',        'Mice': '3',  'Experiment': 'GAS'},
    'S10_CH25H_1': {'orig.ident': 'S10_CH25H', 'Sample': 'S10', 'Type': 'CH25H_KO',  'Mice': '1',  'Experiment': 'GAS'},
    'S11_CH25H_2': {'orig.ident': 'S11_CH25H', 'Sample': 'S11', 'Type': 'CH25H_KO',  'Mice': '2',  'Experiment': 'GAS'},
    'S12_CH25H_3': {'orig.ident': 'S12_CH25H', 'Sample': 'S12', 'Type': 'CH25H_KO',  'Mice': '3',  'Experiment': 'GAS'},
    'S13_Trem2_1': {'orig.ident': 'S13_Trem2', 'Sample': 'S13', 'Type': 'TREM2_KO',  'Mice': '1',  'Experiment': 'GAS'},
    'S14_Trem2_2': {'orig.ident': 'S14_Trem2', 'Sample': 'S14', 'Type': 'TREM2_KO',  'Mice': '2',  'Experiment': 'GAS'},
    'S15_Trem2_3': {'orig.ident': 'S15_Trem2', 'Sample': 'S15', 'Type': 'TREM2_KO',  'Mice': '3',  'Experiment': 'GAS'},
}

def get_h5_paths(sample_dir_name):
    """Return (filtered_h5, raw_h5) absolute paths for a sample."""
    base = CH25H_DIR / sample_dir_name / 'count'
    return (
        str((base / 'sample_filtered_feature_bc_matrix.h5').resolve()),
        str((base / 'sample_raw_feature_bc_matrix.h5').resolve()),
    )

# Verify all paths exist
for sname in SAMPLE_META:
    filt, raw = get_h5_paths(sname)
    assert Path(filt).exists(), f'Missing filtered H5: {filt}'
    assert Path(raw).exists(),  f'Missing raw H5: {raw}'
print(f'All {len(SAMPLE_META)} sample H5 paths verified.')

All 15 sample H5 paths verified.


---
## 3. SoupX: Ambient RNA Correction

Runs SoupX per sample using the raw (all droplets) and filtered (cells only) 10x H5 files.
Seurat is used for a quick clustering step required by `autoEstCont`.

In [18]:
def _r_sparse_to_scipy(r_varname):
    """
    Convert an R dgCMatrix (genes x cells) stored in the R global environment
    to a scipy csr_matrix (cells x genes) and return (matrix, gene_names, cell_names).
    """
    data   = np.array(ro.r(f'{r_varname}@x'), dtype=np.float32)
    i      = np.array(ro.r(f'{r_varname}@i'), dtype=np.int32)
    p      = np.array(ro.r(f'{r_varname}@p'), dtype=np.int32)
    nrow, ncol = [int(x) for x in ro.r(f'dim({r_varname})')]
    genes  = list(ro.r(f'rownames({r_varname})'))
    cells  = list(ro.r(f'colnames({r_varname})'))
    mat_genes_x_cells = csc_matrix((data, i, p), shape=(nrow, ncol))
    mat_cells_x_genes = csr_matrix(mat_genes_x_cells.T)
    return mat_cells_x_genes, genes, cells


def run_soupx(sample_name, filtered_h5, raw_h5):
    """
    Run SoupX ambient RNA correction using R.

    Steps:
      1. Load raw (tod) and filtered (toc) 10x H5 files in R.
      2. Subset both to their common genes (raw H5 may have extra probe-level features).
      3. Create a Seurat object and run quick clustering for contamination estimation.
      4. Run autoEstCont (falls back to rho=0.10 if it fails).
      5. Apply adjustCounts with roundToInt=TRUE.

    Returns AnnData (cells x genes) with SoupX-corrected integer counts.
    """
    print(f'  [SoupX] {sample_name} - loading H5 files...')
    ro.r(f"""
    suppressMessages({{
        toc <- Read10X_h5("{filtered_h5}")
        tod <- Read10X_h5("{raw_h5}")
        if (is.list(toc)) toc <- toc[["Gene Expression"]]
        if (is.list(tod)) tod <- tod[["Gene Expression"]]
        # Subset both to common genes in the same order
        # (raw H5 can contain extra probe-level or Flex features)
        common_genes <- intersect(rownames(tod), rownames(toc))
        tod <- tod[common_genes, ]
        toc <- toc[common_genes, ]
        cat("  genes in toc:", nrow(toc), "| genes in tod:", nrow(tod),
            "| barcodes toc:", ncol(toc), "| barcodes tod:", ncol(tod), "\n")
    }})
    """)

    print(f'  [SoupX] {sample_name} - Seurat clustering for contamination estimate...')
    ro.r("""
    suppressMessages({
        sc_obj <- SoupChannel(tod, toc)

        seu <- CreateSeuratObject(counts = toc, min.cells = 3)
        seu <- NormalizeData(seu, verbose = FALSE)
        seu <- FindVariableFeatures(seu, nfeatures = 2000, verbose = FALSE)
        seu <- ScaleData(seu, features = VariableFeatures(seu), verbose = FALSE)
        seu <- RunPCA(seu, npcs = 30, verbose = FALSE)
        seu <- FindNeighbors(seu, dims = 1:20, verbose = FALSE)
        seu <- FindClusters(seu, resolution = 0.5, verbose = FALSE)

        sc_obj <- setClusters(sc_obj, setNames(seu$seurat_clusters, colnames(seu)))

        tryCatch({
            sc_obj <- autoEstCont(sc_obj, verbose = FALSE)
            cat("  estimated rho =", round(sc_obj$fit$rho, 4), "\n")
        }, error = function(e) {
            message("  autoEstCont failed (", conditionMessage(e), "), using rho = 0.10")
            sc_obj <<- setContaminationFraction(sc_obj, 0.10)
        })

        soup_corrected <- adjustCounts(sc_obj, roundToInt = TRUE)
    })
    """)

    mat, genes, cells = _r_sparse_to_scipy('soup_corrected')
    adata = ad.AnnData(
        X   = mat,
        obs = pd.DataFrame({'barcode': cells}, index=[f'{sample_name}_{c}' for c in cells]),
        var = pd.DataFrame(index=genes),
    )
    adata.var_names_make_unique()
    print(f'  [SoupX] {sample_name} - done: {adata.n_obs} cells x {adata.n_vars} genes')
    return adata

---
## 4. scDblFinder: Doublet Detection

In [19]:
def run_scdblfinder(adata, sample_name):
    """
    Run scDblFinder on an AnnData object (cells x genes).

    Passes the count matrix to R as a SingleCellExperiment, runs scDblFinder,
    and annotates adata.obs with:
      - scDblFinder.class  ("singlet" / "doublet")
      - scDblFinder.score
      - scDblFinder.weighted
      - scDblFinder.cxds_score

    Returns the annotated AnnData.
    """
    print(f'  [scDblFinder] {sample_name} - {adata.n_obs} cells...')

    # Convert count matrix to COO for efficient R transfer
    X = csr_matrix(adata.X)
    coo = X.tocoo()

    # Note: R identifiers cannot start with underscore
    ro.globalenv['dbl_x']     = ro.FloatVector(coo.data.tolist())
    ro.globalenv['dbl_i']     = ro.IntVector((coo.row + 1).tolist())  # 1-indexed rows (cells)
    ro.globalenv['dbl_j']     = ro.IntVector((coo.col + 1).tolist())  # 1-indexed cols (genes)
    ro.globalenv['dbl_nrow']  = ro.IntVector([X.shape[0]])
    ro.globalenv['dbl_ncol']  = ro.IntVector([X.shape[1]])
    ro.globalenv['dbl_genes'] = ro.StrVector(adata.var_names.tolist())
    ro.globalenv['dbl_cells'] = ro.StrVector(adata.obs_names.tolist())

    ro.r("""
    suppressMessages({
        # Reconstruct cells x genes, then transpose to genes x cells for SCE
        counts_cx_g <- sparseMatrix(
            i    = dbl_i,
            j    = dbl_j,
            x    = dbl_x,
            dims = c(dbl_nrow[1], dbl_ncol[1])
        )
        counts_gx_c <- t(counts_cx_g)        # genes x cells
        rownames(counts_gx_c) <- dbl_genes
        colnames(counts_gx_c) <- dbl_cells

        sce <- SingleCellExperiment(list(counts = counts_gx_c))
        set.seed(42)
        sce <- scDblFinder(sce, verbose = FALSE)

        dbl_class    <- as.character(sce$scDblFinder.class)
        dbl_score    <- as.numeric(sce$scDblFinder.score)
        dbl_weighted <- as.numeric(sce$scDblFinder.weighted)
        dbl_cxds     <- if ('scDblFinder.cxds_score' %in% colnames(colData(sce))) {
                             as.numeric(sce$scDblFinder.cxds_score)
                         } else {
                             rep(NA_real_, ncol(sce))
                         }
    })
    """)

    adata.obs['scDblFinder.class']      = list(ro.globalenv['dbl_class'])
    adata.obs['scDblFinder.score']      = list(ro.globalenv['dbl_score'])
    adata.obs['scDblFinder.weighted']   = list(ro.globalenv['dbl_weighted'])
    adata.obs['scDblFinder.cxds_score'] = list(ro.globalenv['dbl_cxds'])

    n_dbl  = (adata.obs['scDblFinder.class'] == 'doublet').sum()
    pct    = 100 * n_dbl / adata.n_obs
    print(f'  [scDblFinder] {sample_name} - {n_dbl} doublets ({pct:.1f}%)')
    return adata

---
## 5. QC Filter Helper

In [20]:
def basic_qc_filter(adata, min_genes=100, max_pct_mito=25.0):
    """Filter low-quality cells by gene count and mitochondrial fraction."""
    n_before = adata.n_obs
    adata.var['mt'] = adata.var_names.str.startswith(('mt-', 'MT-'))
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    sc.pp.filter_cells(adata, min_genes=min_genes)
    adata = adata[adata.obs['pct_counts_mt'] < max_pct_mito].copy()
    print(f'  QC: {n_before:,} -> {adata.n_obs:,} cells')
    return adata

---
## 6. Process New Samples

For each of the 15 new samples: SoupX → metadata annotation → scDblFinder → QC filter.

In [21]:
new_adatas = []

for sample_dir_name, meta in SAMPLE_META.items():
    print(f'\n{"="*60}')
    print(f'Processing: {sample_dir_name}  ({meta["Type"]})')
    print(f'{"="*60}')

    filtered_h5, raw_h5 = get_h5_paths(sample_dir_name)

    # 1. SoupX ambient correction
    adata = run_soupx(sample_dir_name, filtered_h5, raw_h5)

    # 2. Attach sample metadata
    for col, val in meta.items():
        adata.obs[col] = val

    # 3. scDblFinder doublet detection
    adata = run_scdblfinder(adata, sample_dir_name)

    # 4. Basic QC
    adata = basic_qc_filter(adata)

    new_adatas.append(adata)
    print(f'  Retained: {adata.n_obs:,} cells')

print(f'\nFinished processing {len(new_adatas)} new samples.')


Processing: S1_81_WT_1  (WT)
  [SoupX] S1_81_WT_1 - loading H5 files...
  genes in toc: 19059 | genes in tod: 19059 | barcodes toc: 4862 | barcodes tod: 330367 
  [SoupX] S1_81_WT_1 - Seurat clustering for contamination estimate...


R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :  
R callback write-console: 
   
R callback write-console:  'giveCsparse' is deprecated; setting repr="T" for you
  


  [SoupX] S1_81_WT_1 - done: 4862 cells x 19059 genes
  [scDblFinder] S1_81_WT_1 - 4862 cells...
  [scDblFinder] S1_81_WT_1 - 377 doublets (7.8%)
  QC: 4,862 -> 4,854 cells
  Retained: 4,854 cells

Processing: S2_83_WT_2  (WT)
  [SoupX] S2_83_WT_2 - loading H5 files...
  genes in toc: 19059 | genes in tod: 19059 | barcodes toc: 4861 | barcodes tod: 422299 
  [SoupX] S2_83_WT_2 - Seurat clustering for contamination estimate...


R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :  
R callback write-console: 
   
R callback write-console:  'giveCsparse' is deprecated; setting repr="T" for you
  


  [SoupX] S2_83_WT_2 - done: 4861 cells x 19059 genes
  [scDblFinder] S2_83_WT_2 - 4861 cells...
  [scDblFinder] S2_83_WT_2 - 371 doublets (7.6%)
  QC: 4,861 -> 4,839 cells
  Retained: 4,839 cells

Processing: S3_91_WT_3  (WT)
  [SoupX] S3_91_WT_3 - loading H5 files...
  genes in toc: 19059 | genes in tod: 19059 | barcodes toc: 5378 | barcodes tod: 480441 
  [SoupX] S3_91_WT_3 - Seurat clustering for contamination estimate...


...

  [SoupX] S13_Trem2_1 - done: 2870 cells x 19059 genes
  [scDblFinder] S13_Trem2_1 - 2870 cells...
  [scDblFinder] S13_Trem2_1 - 132 doublets (4.6%)
  QC: 2,870 -> 2,838 cells
  Retained: 2,838 cells

Processing: S14_Trem2_2  (TREM2_KO)
  [SoupX] S14_Trem2_2 - loading H5 files...
  genes in toc: 19059 | genes in tod: 19059 | barcodes toc: 3243 | barcodes tod: 245918 
  [SoupX] S14_Trem2_2 - Seurat clustering for contamination estimate...


R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :  
R callback write-console: 
   
R callback write-console:  'giveCsparse' is deprecated; setting repr="T" for you
  


  [SoupX] S14_Trem2_2 - done: 3243 cells x 19059 genes
  [scDblFinder] S14_Trem2_2 - 3243 cells...
  [scDblFinder] S14_Trem2_2 - 200 doublets (6.2%)
  QC: 3,243 -> 3,211 cells
  Retained: 3,211 cells

Processing: S15_Trem2_3  (TREM2_KO)
  [SoupX] S15_Trem2_3 - loading H5 files...
  genes in toc: 19059 | genes in tod: 19059 | barcodes toc: 4471 | barcodes tod: 324000 
  [SoupX] S15_Trem2_3 - Seurat clustering for contamination estimate...


R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :  
R callback write-console: 
   
R callback write-console:  'giveCsparse' is deprecated; setting repr="T" for you
  


  [SoupX] S15_Trem2_3 - done: 4471 cells x 19059 genes
  [scDblFinder] S15_Trem2_3 - 4471 cells...
  [scDblFinder] S15_Trem2_3 - 329 doublets (7.4%)
  QC: 4,471 -> 4,436 cells
  Retained: 4,436 cells

Finished processing 15 new samples.


---
## 7. Load Existing COL5A1 WT/CKO Cells

Extract **S15_82_WT** and **S16_84_CKO** from `xxo_xxt_xyt_xyo_cko_wt` files.
These cells already have scDblFinder annotations from a prior run.

**Gene name harmonization**: the old file used `pd.Index.make_unique()` which produces
`.1` suffixes for duplicates, while anndata uses `-1`.  We rename the 5 affected genes
so the two datasets share a common gene index.

In [25]:
OLD_KEEP = ['S15_82_WT', 'S16_84_CKO']

print('Loading existing xxo_xxt_xyt_xyo_cko_wt files...')
meta_old = pd.read_csv(DATA_DIR / 'metadata_xxo_xxt_xyt_xyo_cko_wt.csv', index_col=0)
print(f'  Full metadata: {len(meta_old):,} cells')
print(f'  Keeping: {OLD_KEEP}')

mask_old   = meta_old['orig.ident'].isin(OLD_KEEP)
meta_keep  = meta_old[mask_old].copy()
keep_idx   = np.where(mask_old.values)[0]   # column indices in genes-x-cells matrix
print(f'  Selected: {len(meta_keep):,} cells')
print(meta_keep['orig.ident'].value_counts())

# Load gene names (may contain .1 suffixes from pandas make_unique)
old_genes = pd.read_csv(
    DATA_DIR / 'gene_names_xxo_xxt_xyt_xyo_cko_wt.csv',
    header=None, names=['gene']
)['gene'].tolist()

# Harmonize .1 -> -1 suffix to match anndata's var_names_make_unique convention
old_genes_harmonized = [
    re.sub(r'\.([0-9]+)$', r'-\1', g) for g in old_genes
]
n_renamed = sum(a != b for a, b in zip(old_genes, old_genes_harmonized))
print(f'  Gene names harmonized (.1 -> -1 suffix): {n_renamed} genes renamed')

# Load full count matrix (genes x cells) and slice columns for kept cells
print('  Loading count matrix (may take a moment)...')
X_full = sio.mmread(str(DATA_DIR / 'counts_xxo_xxt_xyt_xyo_cko_wt.mtx')).tocsc()
print(f'  Full matrix: {X_full.shape[0]} genes x {X_full.shape[1]} cells')

X_keep = csr_matrix(X_full[:, keep_idx].T)  # cells x genes
print(f'  Sliced matrix: {X_keep.shape[0]} cells x {X_keep.shape[1]} genes')

adata_old = ad.AnnData(
    X   = X_keep,
    obs = meta_keep.copy(),
    var = pd.DataFrame(index=old_genes_harmonized),
)

# Add missing metadata columns consistent with new samples
adata_old.obs['Experiment'] = 'COL5A1_KO'
type_map = {'S15_82_WT': 'WT', 'S16_84_CKO': 'COL5A1_KO'}
adata_old.obs['Type'] = adata_old.obs['orig.ident'].map(type_map)

# Keep only the columns present in new data for a clean concat
old_keep_cols = [
    'orig.ident', 'Sample', 'Type', 'Mice', 'Experiment',
    'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score',
]
available = [c for c in old_keep_cols if c in adata_old.obs.columns]
adata_old.obs = adata_old.obs[available]

print(f'\nOld WT/CKO AnnData: {adata_old.n_obs:,} cells x {adata_old.n_vars:,} genes')
print(adata_old.obs['orig.ident'].value_counts())

Loading existing xxo_xxt_xyt_xyo_cko_wt files...
  Full metadata: 98,317 cells
  Keeping: ['S15_82_WT', 'S16_84_CKO']
  Selected: 8,753 cells
orig.ident
S15_82_WT     5395
S16_84_CKO    3358
Name: count, dtype: int64
  Gene names harmonized (.1 -> -1 suffix): 17 genes renamed
  Loading count matrix (may take a moment)...
  Full matrix: 19059 genes x 98317 cells
  Sliced matrix: 8753 cells x 19059 genes

Old WT/CKO AnnData: 8,753 cells x 19,059 genes
orig.ident
S15_82_WT     5395
S16_84_CKO    3358
Name: count, dtype: int64


---
## 8. Combine All Samples

In [26]:
# Drop scanpy QC columns from new adatas before concat (they vary per sample)
qc_drop = ['mt', 'n_genes_by_counts', 'total_counts', 'total_counts_mt',
           'pct_counts_mt', 'log1p_total_counts', 'log1p_n_genes_by_counts',
           'barcode']
for adata in new_adatas:
    drop_obs = [c for c in qc_drop if c in adata.obs.columns]
    adata.obs = adata.obs.drop(columns=drop_obs)
    drop_var = [c for c in qc_drop if c in adata.var.columns]
    adata.var = adata.var.drop(columns=drop_var)

all_adatas = new_adatas + [adata_old]

print('Concatenating all samples...')
adata_combined = ad.concat(
    all_adatas,
    join='outer',       # keep all genes; fill missing with 0
    fill_value=0,
)
adata_combined.obs_names_make_unique()
adata_combined.var_names_make_unique()

print(f'\nCombined: {adata_combined.n_obs:,} cells x {adata_combined.n_vars:,} genes')
print('\nCells per sample:')
print(adata_combined.obs['orig.ident'].value_counts().sort_index())
print('\nCells per type:')
print(adata_combined.obs['Type'].value_counts())
print('\nCells per experiment:')
print(adata_combined.obs['Experiment'].value_counts())

Concatenating all samples...

Combined: 71,318 cells x 19,076 genes

Cells per sample:
orig.ident
S10_CH25H     2465
S11_CH25H     2019
S12_CH25H     2497
S13_Trem2     2838
S14_Trem2     3211
S15_82_WT     5395
S15_Trem2     4436
S16_84_CKO    3358
S1_81_WT      4854
S2_83_WT      4839
S3_91_WT      5341
S4_00_CKO     6813
S5_82_CKO     8421
S6_96_CKO     5856
S7_WT         3386
S8_WT         2005
S9_WT         3584
Name: count, dtype: int64

Cells per type:
Type
WT           29404
COL5A1_KO    24448
TREM2_KO     10485
CH25H_KO      6981
Name: count, dtype: int64

Cells per experiment:
Experiment
COL5A1_KO    44877
GAS          26441
Name: count, dtype: int64


---
## 9. Export to Internal Format

Writes three files matching the project's internal dataset convention:
- `counts_ch25h_col5a1_trem2.mtx` — sparse matrix, **genes x cells**
- `gene_names_ch25h_col5a1_trem2.csv` — one gene per line, no header
- `metadata_ch25h_col5a1_trem2.csv` — R-style CSV with cell barcodes as index

In [27]:
DATASET_NAME = 'ch25h_col5a1_trem2'

# 1. counts MTX (genes x cells)
counts_path = DATA_DIR / f'counts_{DATASET_NAME}.mtx'
X = adata_combined.X
if not isinstance(X, csr_matrix):
    X = csr_matrix(X)
sio.mmwrite(str(counts_path), X.T)  # transpose: genes x cells
print(f'Wrote {counts_path.name}  ({adata_combined.n_vars} genes x {adata_combined.n_obs} cells)')

# 2. gene names (no header)
genes_path = DATA_DIR / f'gene_names_{DATASET_NAME}.csv'
with open(genes_path, 'w') as f:
    for g in adata_combined.var_names:
        f.write(f'{g}\n')
print(f'Wrote {genes_path.name}  ({len(adata_combined.var_names)} genes)')

# 3. metadata (R-style, cell barcodes as index)
meta_path = DATA_DIR / f'metadata_{DATASET_NAME}.csv'
meta_out = adata_combined.obs.copy()
# Drop any remaining scanpy internal columns
drop_internal = [c for c in meta_out.columns if c.startswith(('n_genes', 'n_cells', 'total_counts', 'pct_counts', 'log1p'))]
meta_out = meta_out.drop(columns=[c for c in drop_internal if c in meta_out.columns])
meta_out.to_csv(meta_path)
print(f'Wrote {meta_path.name}  ({len(meta_out)} cells, columns: {list(meta_out.columns)})')

Wrote counts_ch25h_col5a1_trem2.mtx  (19076 genes x 71318 cells)
Wrote gene_names_ch25h_col5a1_trem2.csv  (19076 genes)
Wrote metadata_ch25h_col5a1_trem2.csv  (71318 cells, columns: ['orig.ident', 'Sample', 'Type', 'Mice', 'Experiment', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score'])


---
## 10. Summary

In [28]:
print('=' * 65)
print('CH25H-COL5A1-TREM2 Preprocessing Complete')
print('=' * 65)

for label, path in [
    ('counts MTX',   counts_path),
    ('gene names',   genes_path),
    ('metadata CSV', meta_path),
]:
    size_mb = path.stat().st_size / 1e6
    print(f'  {label}: {path.name}  ({size_mb:.1f} MB)')

print()
print(f'Total cells : {adata_combined.n_obs:,}')
print(f'Total genes : {adata_combined.n_vars:,}')

print()
print('Doublet summary (all samples):')
if 'scDblFinder.class' in adata_combined.obs.columns:
    dbl_summary = (
        adata_combined.obs
        .groupby(['orig.ident', 'scDblFinder.class'])
        .size()
        .unstack(fill_value=0)
        .assign(pct_doublet=lambda df: 100 * df.get('doublet', 0) / (df.sum(axis=1)))
    )
    print(dbl_summary.round(1).to_string())

print()
print('Next steps:')
print('  - Optionally filter doublets: adata = adata[adata.obs["scDblFinder.class"] == "singlet"]')
print('  - Load this dataset in 01_scvi_integration.ipynb via DATASET_CONFIG')
print('=' * 65)

CH25H-COL5A1-TREM2 Preprocessing Complete
  counts MTX: counts_ch25h_col5a1_trem2.mtx  (2449.4 MB)
  gene names: gene_names_ch25h_col5a1_trem2.csv  (0.1 MB)
  metadata CSV: metadata_ch25h_col5a1_trem2.csv  (9.7 MB)

Total cells : 71,318
Total genes : 19,076

Doublet summary (all samples):
scDblFinder.class  doublet  singlet  pct_doublet
orig.ident                                      
S10_CH25H              136     2329          5.5
S11_CH25H              103     1916          5.1
S12_CH25H              134     2363          5.4
S13_Trem2              132     2706          4.7
S14_Trem2              198     3013          6.2
S15_82_WT                0     5395          0.0
S15_Trem2              328     4108          7.4
S16_84_CKO               0     3358          0.0
S1_81_WT               377     4477          7.8
S2_83_WT               371     4468          7.7
S3_91_WT               382     4959          7.2
S4_00_CKO              613     6200          9.0
S5_82_CKO              8